##PART 8 -- UNITY CATALOG
Tasks

1. Create a catalog
2. Create a schema
3. Create a table inside schema
4. Load data into the table

In [0]:
spark.sql("USE CATALOG my_catalog")


DataFrame[]

In [0]:
spark.sql("USE SCHEMA my_schema")

DataFrame[]

In [0]:
spark.sql("""
CREATE TABLE students (
    id INT,
    name STRING,
    age INT
)
""")

DataFrame[]

In [0]:
spark.sql("""
INSERT INTO students VALUES
(1, 'John', 20),
(2, 'Alice', 22),
(3, 'Bob', 21)
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("SELECT * FROM students").show()

+---+-----+---+
| id| name|age|
+---+-----+---+
|  1| John| 20|
|  2|Alice| 22|
|  3|  Bob| 21|
+---+-----+---+



##PART 9 — DATA GOVERNANCE
Tasks 
1. Discover tables using catalog 
2. Create a derived table 
3. Observe lineage 
4. Apply access control 
5. Query audit logs

In [0]:
spark.sql("SHOW TABLES IN my_catalog.my_schema").show()

+---------+---------+-----------+
| database|tableName|isTemporary|
+---------+---------+-----------+
|my_schema| students|      false|
+---------+---------+-----------+



In [0]:
spark.sql("""
CREATE OR REPLACE TABLE my_catalog.my_schema.students_adults AS
SELECT * FROM my_catalog.my_schema.students
WHERE age >= 21
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("SELECT * FROM my_catalog.my_schema.students_adults").show()

+---+-----+---+
| id| name|age|
+---+-----+---+
|  2|Alice| 22|
|  3|  Bob| 21|
+---+-----+---+



In [0]:
spark.sql("""
GRANT SELECT ON TABLE my_catalog.my_schema.students 
TO `azuser6410_mml.local@karthikirisoutlook.onmicrosoft.com`
""")

DataFrame[]

In [0]:
spark.sql("SHOW GRANTS ON TABLE my_catalog.my_schema.students").show()

+--------------------+----------+----------+--------------------+
|           Principal|ActionType|ObjectType|           ObjectKey|
+--------------------+----------+----------+--------------------+
|azuser6410_mml.lo...|    SELECT|     TABLE|my_catalog.my_sch...|
+--------------------+----------+----------+--------------------+



##FINAL CAPSTONE
Build a system that:
Raw Data → Clean Data → Analytics → Governed Data
Requirements
- Use Delta tables
- Use incremental logic
- Build DLT pipeline
- Create a Simple Unity Catalog structure

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS my_catalog.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS my_catalog.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS my_catalog.gold")

DataFrame[]

In [0]:
data = [
    (1, "John", 20),
    (2, "Alice", 22),
    (3, "Bob", None),
    (4, "Eve", 25)
]
df = spark.createDataFrame(data, ["id", "name", "age"])
df.write.mode("overwrite").saveAsTable("my_catalog.bronze.students_raw")

In [0]:
spark.sql("SELECT * FROM my_catalog.bronze.students_raw").show()

+---+-----+----+
| id| name| age|
+---+-----+----+
|  1| John|  20|
|  2|Alice|  22|
|  3|  Bob|NULL|
|  4|  Eve|  25|
+---+-----+----+



In [0]:
spark.sql("""
CREATE OR REPLACE TABLE my_catalog.silver.students_clean AS
SELECT * FROM my_catalog.bronze.students_raw
WHERE age IS NOT NULL
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("SELECT * FROM my_catalog.silver.students_clean").show()

+---+-----+---+
| id| name|age|
+---+-----+---+
|  1| John| 20|
|  2|Alice| 22|
|  4|  Eve| 25|
+---+-----+---+



In [0]:
spark.sql("""
CREATE OR REPLACE TABLE my_catalog.gold.students_summary AS
SELECT 
    COUNT(*) AS total_students,
    AVG(age) AS avg_age
FROM my_catalog.silver.students_clean
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("SELECT * FROM my_catalog.gold.students_summary").show()

+--------------+------------------+
|total_students|           avg_age|
+--------------+------------------+
|             3|22.333333333333332|
+--------------+------------------+



In [0]:
spark.sql("""
MERGE INTO my_catalog.silver.students_clean AS target
USING my_catalog.bronze.students_raw AS source
ON target.id = source.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("SELECT * FROM my_catalog.silver.students_clean").show()

+---+-----+---+
| id| name|age|
+---+-----+---+
|  1| John| 20|
|  2|Alice| 22|
|  4|  Eve| 25|
+---+-----+---+



In [0]:
spark.sql("DESCRIBE DETAIL my_catalog.silver.students_clean").show()

+------+--------------------+--------------------+-----------+--------+--------------------+-------------------+----------------+-----------------+--------+-----------+--------------------+----------------+----------------+--------------------+--------------------+-------------+
|format|                  id|                name|description|location|           createdAt|       lastModified|partitionColumns|clusteringColumns|numFiles|sizeInBytes|          properties|minReaderVersion|minWriterVersion|       tableFeatures|          statistics|clusterByAuto|
+------+--------------------+--------------------+-----------+--------+--------------------+-------------------+----------------+-----------------+--------+-----------+--------------------+----------------+----------------+--------------------+--------------------+-------------+
| delta|7dcaba39-e7be-47b...|my_catalog.silver...|       NULL|        |2026-05-04 05:39:...|2026-05-04 05:40:16|              []|               []|       1|    

In [0]:
display(spark.sql("SELECT * FROM my_catalog.bronze.students_raw"))
display(spark.sql("SELECT * FROM my_catalog.silver.students_clean"))
display(spark.sql("SELECT * FROM my_catalog.gold.students_summary"))

id,name,age
1,John,20
2,Alice,22
3,Bob,null
4,Eve,25


id,name,age
1,John,20
2,Alice,22
4,Eve,25


total_students,avg_age
3,22.333333333333332
